# CAD Data Cleaning & Preparation
**Project:** Investigating Crisis Call Volumes — Eugene vs. Springfield  
**Author:** Ilhan Haniff  
**Description:** This notebook imports, cleans, and standardizes all Eugene CAD CSV files from 2015–2025, constructs temporal and agency classification features, and exports a single combined cleaned dataset for analysis.

**Input:** Raw yearly CAD CSV files (`data/EugeneCAD{year}noloc.csv`)  
**Output:** `data/cad_clean_2015_2025.csv`

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os

## 2. Load Raw Data
Import each yearly CAD file and combine into one raw DataFrame.

In [3]:
years = range(2015, 2026)
data_folder = 'data'

raw_frames = []

for year in years:
    filepath = os.path.join(data_folder, f'EugeneCAD{year}noloc.csv')
    df = pd.read_csv(filepath, low_memory=False)
    raw_frames.append(df)
    print(f'{year}: {df.shape[0]:,} rows loaded')

cad_raw = pd.concat(raw_frames, ignore_index=True)
print(f'\nTotal combined rows: {cad_raw.shape[0]:,}')
print(f'Columns: {list(cad_raw.columns)}')

2015: 123,248 rows loaded
2016: 127,732 rows loaded
2017: 129,688 rows loaded
2018: 127,432 rows loaded
2019: 137,089 rows loaded
2020: 137,287 rows loaded
2021: 137,527 rows loaded
2022: 140,545 rows loaded
2023: 135,464 rows loaded
2024: 132,650 rows loaded
2025: 117,352 rows loaded

Total combined rows: 1,446,014
Columns: ['yr', 'agency', 'service', 'inci_id', 'calltime', 'case_id', 'callsource', 'nature', 'closecode', 'closed_as', 'secs_to_disp', 'secs_to_arrv', 'secs_to_close', 'disp', 'arrv', 'priority', 'primeunit', 'units_dispd', 'units_arrived', 'month']


## 3. Initial Inspection

In [6]:
# First few rows
cad_raw.head()

,yr,agency,service,inci_id,calltime,case_id,callsource,nature,closecode,closed_as,secs_to_disp,secs_to_arrv,secs_to_close,disp,arrv,priority,primeunit,units_dispd,units_arrived,month
0,2015,EPD,LAW,15000001,2015-01-01 00:00:00.000,NaN,SELF,PERSON STOP,ASST,ASSISTED,0.0,0.0,217,1,1,6,_5E48,1,1,NaN
1,2015,EPD,LAW,15000002,2015-01-01 00:00:44.000,NaN,SELF,FIGHT,RSLV,RESOLVED,0.0,0.0,2114,1,1,P,_3F65,4,2,NaN
2,2015,EPD,LAW,15000003,2015-01-01 00:01:05.000,NaN,PHONE,CHECK WELFARE,ASST,ASSISTED,708.0,1094.0,1538,1,1,5,_3J79,1,1,NaN
3,2015,EPD,LAW,15000007,2015-01-01 00:03:16.000,NaN,W911,SHOTS FIRED,PCHK,PATROL CHECK,277.0,576.0,891,1,1,3,_5E48,2,2,NaN
4,2015,EPD,LAW,15000010,2015-01-01 00:03:34.000,NaN,SELF,ILLEGAL FIREWORKS,ADVI,ADVISED,0.0,0.0,150,1,1,5,_5K97,1,1,NaN


In [7]:
# Data types
cad_raw.dtypes

yr                 int64
agency            object
service           object
inci_id            int64
calltime          object
case_id           object
callsource        object
nature            object
closecode         object
closed_as         object
secs_to_disp     float64
secs_to_arrv     float64
secs_to_close      int64
disp               int64
arrv               int64
priority          object
primeunit         object
units_dispd        int64
units_arrived      int64
month            float64
dtype: object

In [8]:
# Missing value counts per column
cad_raw.isnull().sum()

yr                     0
agency                 0
service                0
inci_id                0
calltime               0
case_id          1213169
callsource             5
nature                53
closecode              0
closed_as          23928
secs_to_disp      414622
secs_to_arrv      532540
secs_to_close          0
disp                   0
arrv                   0
priority               0
primeunit         354283
units_dispd            0
units_arrived          0
month            1328662
dtype: int64

## 4. Select Relevant Columns
Keep only the columns relevant to the research questions.

In [45]:
cols_to_keep = ['agency', 'inci_id', 'calltime', 'primeunit']

cad = cad_raw[cols_to_keep].copy()
print(f'New Shape: {cad.shape}')

New Shape: (1446014, 4)


In [46]:
# First few rows
cad.head(5)

,agency,inci_id,calltime,primeunit
0,EPD,15000001,2015-01-01 00:00:00.000,_5E48
1,EPD,15000002,2015-01-01 00:00:44.000,_3F65
2,EPD,15000003,2015-01-01 00:01:05.000,_3J79
3,EPD,15000007,2015-01-01 00:03:16.000,_5E48
4,EPD,15000010,2015-01-01 00:03:34.000,_5K97


## 5. Standardize Variable Names and Text
Rename columns for consistency, strip whitespace from all string columns.

In [47]:
# Strip leading/trailing whitespace from all string columns
str_cols = cad.select_dtypes(include='object').columns
for col in str_cols:
    cad[col] = cad[col].str.strip()

# Standardize text to uppercase for consistency
for col in ['agency', 'primeunit']:
    cad[col] = cad[col].str.upper()

print('String columns cleaned:', list(str_cols))
print('Unique agency values after cleaning:', cad['agency'].unique())

String columns cleaned: ['agency', 'calltime', 'primeunit']
Unique agency values after cleaning: ['EPD' 'CAHE']


## 6. Convert calltime to Datetime and Extract Temporal Features

In [48]:
# Convert calltime to datetime
cad['calltime'] = pd.to_datetime(cad['calltime'], errors='coerce')

# Report any rows where conversion failed
failed = cad['calltime'].isna().sum()
print(f'Rows with missing calltime: {failed}')

# Extract temporal features
cad['call_year']  = cad['calltime'].dt.year
cad['call_month'] = cad['calltime'].dt.month
cad['call_day']   = cad['calltime'].dt.day
cad['call_hour']  = cad['calltime'].dt.hour
cad['call_dow']   = cad['calltime'].dt.day_name()  # day of week

cad.head(5)

Rows with missing calltime: 0


,agency,inci_id,calltime,primeunit,call_year,call_month,call_day,call_hour,call_dow
0,EPD,15000001,2015-01-01 00:00:00,_5E48,2015,1,1,0,Thursday
1,EPD,15000002,2015-01-01 00:00:44,_3F65,2015,1,1,0,Thursday
2,EPD,15000003,2015-01-01 00:01:05,_3J79,2015,1,1,0,Thursday
3,EPD,15000007,2015-01-01 00:03:16,_5E48,2015,1,1,0,Thursday
4,EPD,15000010,2015-01-01 00:03:34,_5K97,2015,1,1,0,Thursday


## 7. Classify Unit Type (agency classification)

The `agency` column explicitly identifies CAHOOTS only from 2022 onward (`'CAHE'`). For earlier years (2015–2021), CAHOOTS units are identified by `'J'` in the `primeunit` field (e.g., `_3J79`). All other units are classified as EPD or other.

Classification rules:
- `agency == 'CAHE'` → `'CAHOOTS'`
- `agency == 'EPD'` AND `'J'` in primeunit → `'CAHOOTS'`
- Missing primeunit → `'UNKNOWN'`
- All remaining EPD units → `'EPD'`

In [57]:
# Preview primeunit and agency patterns to inform classification
print('Sample primeunit values:')
print(cad['primeunit'].dropna().unique()[:1000])
print('Sample agency values:')
print(cad['agency'].dropna().unique()[:20])

Sample primeunit values:
['_5E48' '_3F65' '_3J79' '_5K97' '_4E31' '_4X40' '_5E57' '_3F68' '_1E53'
 '_4E81' '_6E28' '_6E97' '_4U71' '_6E15' '_6B36' '_AWO' '_5E58' '_4B51'
 '_5E65' '_5B44' '_4B11' '_5B62' '_5T84' '_CMD8' '_CAHOT' '_4X41' '_4E54'
 '_5B81' '_4B61' '_6X60' '_6T86' '_4F72' '_2U26' '_1F64' '_1E44' '_1M13'
 '_2E34' '_1E57' '_3E46' '_2E14' '_2Y35' '_3E48' '_3J78' '_3E55' '_3E18'
 '_3E58' '_3F67' '_3E29' '_3E39' '_3E25' '_1M15' '_3E16' '_2K81' '_4T88'
 '_4E91' '_4E61' '_4E12' '_4E51' '_4E42' '_3X30' '_4F73' '_6E16' '_5E88'
 '_CMD9' '_5E66' '_5E46' '_6E29' '_6E38' '_5E55' '_1D12' '_1E67' '_1E51'
 '_2C82' '_2R17' '_2Y16' '_3W73' '_CNTR' '_3F61' '_2Y12' '_3W79' '_4E53'
 '_5T83' '_SLT1' '_2X20' '_4E93' '_4E52' '_5T82' '_5T81' '_4E13' '_1E54'
 '_1E43' '_3U38' '_2E21' '_2E94' '_3E19' '_1E69' '_2E92' '_3E49' '_2C81'
 '_2E35' '_3E26' '_3E68' '_3V78' '_3W74' '_3E45' '_3E59' '_3E56' '_CMD2'
 '_3X90' '_4E43' '_4E11' '_5E51' '_5E62' '_2R13' '_5E47' '_6E35' '_6E26'
 '_6E19' '_1F66' '_1X11' '

In [63]:
def classify_unit(row):
    """
    Classifies each call's responding unit type based on agency and primeunit.
    
    Returns one of: 'CAHOOTS', 'EPD', or 'UNKNOWN'
    """
    agency = str(row['agency']).strip() if pd.notna(row['agency']) else ''
    unit_missing = pd.isna(row['primeunit'])
    unit = str(row['primeunit']).strip() if not unit_missing else ''

    # Missing primeunit — cannot classify
    if unit_missing:
        return 'UNKNOWN'

    
    # Explicit CAHOOTS agency label (2022+)
    if agency == 'CAHE':
        return 'CAHOOTS'

    # CAHOOTS identified by 'J' in primeunit (pre-2022)
    if agency == 'EPD' and 'J' in unit:
        return 'CAHOOTS'

    # All remaining EPD units
    return 'EPD'

cad['unit_type'] = cad.apply(classify_unit, axis=1)
cad = cad.drop(columns='agency')

print('unit_type value counts:')
print(cad['unit_type'].value_counts())

unit_type value counts:
unit_type
EPD        933692
UNKNOWN    354283
CAHOOTS    158039
Name: count, dtype: int64


In [64]:
# Verify CAHOOTS classification by year — should be > 0 for 2015-2021
print('CAHOOTS calls by year:')
print(cad[cad['unit_type'] == 'CAHOOTS'].groupby('call_year').size())

CAHOOTS calls by year:
call_year
2015     9547
2016    10989
2017    14122
2018    15055
2019    16414
2020    16534
2021    16746
2022    20317
2023    17354
2024    16532
2025     4429
dtype: int64


## 8. Check for Duplicates and Missing Values

In [65]:
# Check for duplicate incident IDs
dupes = cad.duplicated(subset=['inci_id']).sum()
print(f'Duplicate inci_id rows: {dupes}')

# Missing values summary
print('\nMissing values per column:')
print(cad.isnull().sum())

Duplicate inci_id rows: 0

Missing values per column:
inci_id            0
calltime           0
primeunit     354283
call_year          0
call_month         0
call_day           0
call_hour          0
call_dow           0
unit_type          0
dtype: int64


In [66]:
# Note: primeunit missing values are kept — some calls may not have a unit assigned.
# These will show as 'UNKNOWN' in unit_type

print(f"Rows with missing primeunit: {cad['primeunit'].isna().sum()}")
print(f"Rows classified as UNKNOWN: {(cad['unit_type'] == 'UNKNOWN').sum()}")

Rows with missing primeunit: 354283
Rows classified as UNKNOWN: 354283


## 9. Final Dataset Summary

In [67]:
print('=== Final Cleaned Dataset Summary ===')
print(f'Shape: {cad.shape}')
print(f'Date range: {cad["calltime"].min()} to {cad["calltime"].max()}')
print(f'\nColumns: {list(cad.columns)}')
print(f'\nunit_type distribution:')
print(cad['unit_type'].value_counts())
print(f'\nCalls per year:')
print(cad.groupby('call_year').size())

=== Final Cleaned Dataset Summary ===
Shape: (1446014, 9)
Date range: 2015-01-01 00:00:00 to 2025-12-31 23:56:03

Columns: ['inci_id', 'calltime', 'primeunit', 'call_year', 'call_month', 'call_day', 'call_hour', 'call_dow', 'unit_type']

unit_type distribution:
unit_type
EPD        933692
UNKNOWN    354283
CAHOOTS    158039
Name: count, dtype: int64

Calls per year:
call_year
2015    123248
2016    127732
2017    129688
2018    127432
2019    137089
2020    137287
2021    137527
2022    140545
2023    135464
2024    132650
2025    117352
dtype: int64


In [68]:
# Preview final dataset
cad.head()

,inci_id,calltime,primeunit,call_year,call_month,call_day,call_hour,call_dow,unit_type
0,15000001,2015-01-01 00:00:00,_5E48,2015,1,1,0,Thursday,EPD
1,15000002,2015-01-01 00:00:44,_3F65,2015,1,1,0,Thursday,EPD
2,15000003,2015-01-01 00:01:05,_3J79,2015,1,1,0,Thursday,CAHOOTS
3,15000007,2015-01-01 00:03:16,_5E48,2015,1,1,0,Thursday,EPD
4,15000010,2015-01-01 00:03:34,_5K97,2015,1,1,0,Thursday,EPD


## 10. Export Cleaned Dataset

In [69]:
output_path = os.path.join(data_folder, 'cad_clean_2015_2025.csv')
cad.to_csv(output_path, index=False)
print(f'Cleaned dataset saved to: {output_path}')
print(f'Final shape: {cad.shape}')

Cleaned dataset saved to: data/cad_clean_2015_2025.csv
Final shape: (1446014, 9)
